# Titanic Survival Prediction

**Kaggle — Titanic: Machine Learning from Disaster**

This project builds an end-to-end binary classification workflow for predicting Titanic passenger survival. The notebook emphasizes reproducible preprocessing, exploratory analysis, feature engineering, cross-validation, model comparison, hyperparameter tuning, and model interpretation.

### Project highlights
- Established a simple rule-based baseline before fitting ML models.
- Explored survival patterns across sex, passenger class, age, fare, family structure, cabin, embarkation port, and ticket groups.
- Evaluated engineered features with stratified 5-fold cross-validation rather than relying on a single split.
- Compared Logistic Regression, Random Forest, and XGBoost.
- Tuned tree-based models with `GridSearchCV`.
- Interpreted the final Random Forest using permutation importance and out-of-fold ROC-AUC.
- Produced a Kaggle-ready submission file.

**Best local Random Forest performance:** ~84% 5-fold CV accuracy and ~0.87 out-of-fold ROC-AUC.  
**Initial Kaggle public leaderboard score:** 0.76555.

> The focus of this notebook is the modeling process and the reasoning behind each experiment, not leaderboard optimization.

## 1. Setup and data loading

The Kaggle notebook paths are used by default. If running locally, replace `TRAIN_PATH` and `TEST_PATH` with the locations of the downloaded competition CSV files.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

RANDOM_STATE = 42

TRAIN_PATH = "/kaggle/input/competitions/titanic/train.csv"
TEST_PATH = "/kaggle/input/competitions/titanic/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
train.head()

## 2. Initial data audit

Before modeling, I inspect schema, missing values, summary statistics, and target balance. Titanic is small enough that understanding the individual variables is more useful than immediately applying automated feature selection or dimensionality reduction.

In [ ]:
display(train.describe(include="all").T)

missing = (
    train.isna()
    .sum()
    .to_frame("Missing")
    .assign(Percent=lambda x: 100 * x["Missing"] / len(train))
    .sort_values("Missing", ascending=False)
)

display(missing[missing["Missing"] > 0])

print("\nTarget distribution:")
display(train["Survived"].value_counts(normalize=True).rename("Proportion"))

## 3. Simple baseline

A useful first benchmark is the historical intuition that women were more likely to survive. This deliberately simple rule gives a baseline that later models should improve upon.

In [ ]:
rule_predictions = (train["Sex"] == "female").astype(int)
rule_accuracy = accuracy_score(train["Survived"], rule_predictions)

print(f"Female-survives rule accuracy: {rule_accuracy:.3f}")

## 4. Exploratory data analysis

### Passenger class and sex

Both variables show strong relationships with survival. Passenger class also acts as a proxy for socioeconomic status and is related to fare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(data=train, x="Pclass", y="Survived", ax=axes[0])
axes[0].set_title("Survival Rate by Passenger Class")
axes[0].set_ylabel("Survival Rate")

sns.barplot(data=train, x="Sex", y="Survived", hue="Pclass", ax=axes[1])
axes[1].set_title("Survival Rate by Sex and Class")
axes[1].set_ylabel("Survival Rate")

plt.tight_layout()
plt.show()

display(train.groupby("Pclass")["Survived"].agg(["count", "mean"]))
display(train.groupby("Sex")["Survived"].agg(["count", "mean"]))

### Age

Age contains substantial missingness, so it will be imputed inside the modeling pipeline. For EDA only, age groups help expose nonlinear survival patterns—particularly the different behavior of children.

In [ ]:
eda = train.copy()

eda["AgeGroup"] = pd.cut(
    eda["Age"],
    bins=[0, 12, 18, 30, 50, 80],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"],
)

age_summary = (
    eda.groupby(["Sex", "AgeGroup"], observed=True)["Survived"]
    .agg(["count", "mean"])
)

display(age_summary)

sns.barplot(data=eda, x="AgeGroup", y="Survived", hue="Sex")
plt.title("Survival Rate by Age Group and Sex")
plt.ylabel("Survival Rate")
plt.xticks(rotation=20)
plt.show()

### Fare and family structure

`SibSp` and `Parch` can be combined into `FamilySize`. The EDA suggests that small family groups often did better than passengers traveling alone or in very large groups.

In [ ]:
eda["FamilySize"] = eda["SibSp"] + eda["Parch"] + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=eda, x="Fare", bins=30, ax=axes[0])
axes[0].set_title("Fare Distribution")

sns.barplot(data=eda, x="FamilySize", y="Survived", ax=axes[1])
axes[1].set_title("Survival Rate by Family Size")
axes[1].set_ylabel("Survival Rate")

plt.tight_layout()
plt.show()

display(eda.groupby("FamilySize")["Survived"].agg(["count", "mean"]))

### Cabin, embarkation port, and ticket groups

Cabin availability is highly associated with passenger class, so apparent cabin effects may partly reflect socioeconomic status. Ticket group size is also strongly related to family size. These relationships motivate candidate engineered features, but each feature is evaluated with cross-validation before being kept.

In [ ]:
eda["HasCabin"] = eda["Cabin"].notna().astype(int)
eda["Deck"] = eda["Cabin"].str[0]
eda["TicketGroupSize"] = eda.groupby("Ticket")["Ticket"].transform("count")

print("Survival by embarkation port:")
display(eda.groupby("Embarked")["Survived"].agg(["count", "mean"]))

print("\nSurvival by cabin availability:")
display(eda.groupby("HasCabin")["Survived"].agg(["count", "mean"]))

print("\nSurvival by ticket group size:")
display(eda.groupby("TicketGroupSize")["Survived"].agg(["count", "mean"]))

print("\nFamily size / ticket group size correlation:")
display(eda[["FamilySize", "TicketGroupSize"]].corr())

## 5. Feature engineering

Candidate features are created consistently for both train and test data.

- `FamilySize`: passenger + siblings/spouses + parents/children
- `IsAlone`: whether the passenger traveled without immediate family
- `HasCabin`: whether cabin information is present
- `Deck`: first character of the cabin
- `TicketGroupSize`: number of passengers sharing a ticket
- `TicketPrefix`: normalized non-numeric ticket prefix
- `IsChild`: passenger younger than 16
- `MaleChild`: interaction suggested by the age × sex EDA

These features are hypotheses, not assumptions: their value is tested under cross-validation.

In [ ]:
def add_features(df):
    df = df.copy()

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0]

    # Group sizes must be computed within the supplied dataframe.
    df["TicketGroupSize"] = df.groupby("Ticket")["Ticket"].transform("count")

    df["TicketPrefix"] = (
        df["Ticket"]
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[./]", "", regex=True)
        .str.replace(r"\s+", "", regex=True)
        .replace("", "NONE")
    )

    df["IsChild"] = (df["Age"] < 16).astype(int)
    df["MaleChild"] = (
        (df["Sex"] == "male") & (df["Age"] < 16)
    ).astype(int)

    return df


train_fe = add_features(train)
test_fe = add_features(test)

## 6. Evaluation strategy

Model and feature choices are compared using the **same stratified 5-fold cross-validation splits**. Keeping preprocessing inside each sklearn `Pipeline` prevents imputation, scaling, or one-hot encoding from leaking information across folds.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

base_features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
]

candidate_features = [
    "FamilySize",
    "IsAlone",
    "HasCabin",
    "Deck",
    "TicketGroupSize",
    "TicketPrefix",
    "MaleChild",
]


def build_preprocessor(features, scale_numeric=False):
    numeric = train_fe[features].select_dtypes(include="number").columns.tolist()
    categorical = [col for col in features if col not in numeric]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric),
        ("cat", categorical_pipeline, categorical),
    ])


def cv_accuracy(model, features):
    scores = cross_val_score(
        model,
        train_fe[features],
        train_fe["Survived"],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    return scores.mean(), scores.std()

## 7. Logistic Regression baseline

Logistic Regression provides an interpretable linear baseline. Numeric variables are standardized; categorical variables are imputed and one-hot encoded.

In [ ]:
def make_logistic(features):
    return Pipeline([
        ("preprocessor", build_preprocessor(features, scale_numeric=True)),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])


logistic = make_logistic(base_features)
lr_mean, lr_std = cv_accuracy(logistic, base_features)

print(f"Logistic Regression CV accuracy: {lr_mean:.3f} ± {lr_std:.3f}")

### Feature experiments with Logistic Regression

Each candidate is added to the same baseline independently. This avoids confusing the effect of one engineered feature with another.

In [ ]:
feature_results = []

for feature in ["FamilySize", "IsAlone", "HasCabin", "Deck", "TicketGroupSize", "TicketPrefix", "MaleChild"]:
    features = base_features + [feature]
    model = make_logistic(features)
    mean_score, std_score = cv_accuracy(model, features)

    feature_results.append({
        "Added Feature": feature,
        "Mean CV Accuracy": mean_score,
        "CV Std": std_score,
    })

lr_feature_results = (
    pd.DataFrame(feature_results)
    .sort_values("Mean CV Accuracy", ascending=False)
    .reset_index(drop=True)
)

display(lr_feature_results)

## 8. Random Forest

Random Forest can capture nonlinear effects and interactions without explicitly specifying them. The initial model is followed by a compact grid search over the hyperparameters that had the clearest effect during experimentation.

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(base_features, scale_numeric=False)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_param_grid = {
    "classifier__n_estimators": [100, 200, 500],
    "classifier__max_depth": [6, 8, 10, None],
    "classifier__min_samples_leaf": [1, 2, 5],
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

rf_grid.fit(train_fe[base_features], train_fe["Survived"])

print("Best Random Forest parameters:")
print(rf_grid.best_params_)
print(f"Best Random Forest CV accuracy: {rf_grid.best_score_:.3f}")

## 9. XGBoost

XGBoost is evaluated with the same preprocessing and CV strategy. A focused parameter grid keeps the search interpretable and computationally reasonable for this small dataset.

In [ ]:
xgb_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(base_features, scale_numeric=False)),
    ("classifier", XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    )),
])

xgb_param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [2, 3, 4],
    "classifier__learning_rate": [0.05, 0.1, 0.2],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=xgb_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

xgb_grid.fit(train_fe[base_features], train_fe["Survived"])

print("Best XGBoost parameters:")
print(xgb_grid.best_params_)
print(f"Best XGBoost CV accuracy: {xgb_grid.best_score_:.3f}")

## 10. Model comparison

The models are compared primarily by cross-validated accuracy. Random Forest was selected for the first Kaggle submission because it combined strong CV performance with strong out-of-fold ranking performance.

In [ ]:
model_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Mean CV Accuracy": lr_mean,
        "CV Std": lr_std,
    },
    {
        "Model": "Random Forest",
        "Mean CV Accuracy": rf_grid.best_score_,
        "CV Std": rf_grid.cv_results_["std_test_score"][rf_grid.best_index_],
    },
    {
        "Model": "XGBoost",
        "Mean CV Accuracy": xgb_grid.best_score_,
        "CV Std": xgb_grid.cv_results_["std_test_score"][xgb_grid.best_index_],
    },
]).sort_values("Mean CV Accuracy", ascending=False)

display(model_comparison)

## 11. Out-of-fold evaluation of the selected Random Forest

Out-of-fold probabilities give every training passenger a prediction from a model that was **not trained on that passenger**. This supports a cleaner estimate of ROC-AUC and allows the default 0.50 classification threshold to be checked without relying on one arbitrary train/validation split.

In [ ]:
best_rf = rf_grid.best_estimator_

rf_oof_prob = cross_val_predict(
    best_rf,
    train_fe[base_features],
    train_fe["Survived"],
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

rf_oof_pred = (rf_oof_prob >= 0.50).astype(int)

print(f"OOF accuracy: {accuracy_score(train_fe['Survived'], rf_oof_pred):.3f}")
print(f"OOF ROC-AUC:  {roc_auc_score(train_fe['Survived'], rf_oof_prob):.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(train_fe["Survived"], rf_oof_pred))
print("\nClassification report:")
print(classification_report(train_fe["Survived"], rf_oof_pred))

## 12. Model interpretation

Permutation importance measures how much predictive performance deteriorates when a feature is randomly shuffled. Unlike raw tree impurity importance, it evaluates each original input feature directly.

In this project, **sex** is the dominant predictor, followed by passenger class, age, and fare. The result also reinforces an important modeling lesson: engineered features are only useful if they add predictive information beyond what the model can already learn from the original variables.

In [ ]:
best_rf.fit(train_fe[base_features], train_fe["Survived"])

perm = permutation_importance(
    best_rf,
    train_fe[base_features],
    train_fe["Survived"],
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="accuracy",
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        "Feature": base_features,
        "Importance": perm.importances_mean,
        "Std": perm.importances_std,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(perm_df)

plot_df = perm_df.sort_values("Importance")

plt.figure(figsize=(8, 5))
plt.barh(plot_df["Feature"], plot_df["Importance"])
plt.xlabel("Mean Decrease in Accuracy")
plt.title("Random Forest Permutation Importance")
plt.tight_layout()
plt.show()

## 13. Final training and Kaggle submission

After model selection, the Random Forest is refit on all labeled training rows and used to predict the competition test set. The submission contains exactly the two columns required by Kaggle: `PassengerId` and `Survived`.

In [ ]:
best_rf.fit(train_fe[base_features], train_fe["Survived"])

test_predictions = best_rf.predict(test_fe[base_features]).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_fe["PassengerId"],
    "Survived": test_predictions,
})

print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"Predicted survival rate: {submission['Survived'].mean():.3f}")

submission.to_csv("/kaggle/working/submission_rf.csv", index=False)
print("\nSaved: /kaggle/working/submission_rf.csv")

## 14. Conclusions

This project reinforced several practical lessons:

- **Start simple.** A basic rule and Logistic Regression created useful benchmarks before more complex models were introduced.
- **Use EDA to generate hypotheses.** Features such as family size, ticket groups, deck, and child interactions came from observed patterns rather than arbitrary transformations.
- **Validate feature engineering.** Intuitive features did not always improve cross-validated performance.
- **Keep preprocessing inside the pipeline.** This makes experiments reproducible and prevents leakage across CV folds.
- **Complexity is not automatically better.** XGBoost did not clearly dominate the simpler alternatives on this small dataset.
- **Interpretability matters.** Permutation importance showed that sex, class, age, and fare carried most of the predictive signal.
- **Local validation is imperfect.** The first Kaggle submission scored lower than the local CV estimate, highlighting the risk of repeatedly optimizing against the same validation procedure.

### Potential next experiments
A natural extension would be a soft-voting ensemble of Logistic Regression, Random Forest, and XGBoost, evaluated with the same cross-validation folds before submission. Repeated or nested cross-validation could also provide a more conservative estimate after extensive model selection.